In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):
    
    # 1. Force a clean extraction every time by removing the old temp_dir if it exists
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)
    
    # Extract the archive using the correct path
    print(f"Extracting {input_archive} to {temp_dir}...")
    subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    # 2. Check the folder structure dynamically
    wheels_path = os.path.join(temp_dir, 'wheels')
    if not os.path.exists(wheels_path):
        print(f"Subfolder 'wheels' not found. Defaulting to: {temp_dir}")
        wheels_path = temp_dir
    else:
        print(f"Found wheels in: {wheels_path}")
        
    print("Installing packages...")
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        wheels_path, 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)
    print("Installation complete!")

In [5]:
# Use the exact correct path you provided
set_env(
    input_archive='/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)


Extracting /kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz to /kaggle/tmp/setup...
Found wheels in: /kaggle/tmp/setup/wheels
Installing packages...
Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
tpot 1.1.0 requires matplotlib>=3.6.2, which is not installed.
tpot 1.1.0 requires scikit-learn>=1.6, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.35.0 requires matplotlib>=3.7.1, which is not installed.
giddy 2.3.8 requires matplotlib, which is not installed.
sentence-transformers 5.2.3 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is 

In [6]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [7]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [8]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [9]:
class CFG:
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, `scipy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'

        '# Advanced Mathematics (scipy):\n'
        '- Combinatorics (e.g., scipy.special.comb)\n'
        '- Special mathematical functions\n'
        '- Optimization and root-finding\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    summarizer_system_prompt = (
        "You are an expert mathematical technical writer. Your job is to read a raw "
        "problem-solving trace produced by a code agent and provide a summary. "
        "Do not attempt to write code or re-solve the problem."
        "Your role is to objectively summarize the solution presented in the shared trace."

        "It is very important not to whitewash the solution, since it will be compared "
        "against different proposed solutions. "

        "Summarize how the agent understood the question, the agent’s key hypotheses, "
        "the exact mathematical reasoning, and the core logic from the provided trace. "


        "You may include notes about the agent’s understanding of the problem, its key "
        "assumptions, boundaries of the solution, inclusions, exclusions, and related "
        "details. Be neutral, but make sure to capture the important details of the "
        "solution. Your summary must be fully auditable by the judge, without any confusion. "
        "Do not forget to include code blocks where relevant."
    )

    
    summarizer_reasoning_effort = ReasoningEffort.HIGH
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
    draft_model_path = '/kaggle/input/datasets/khoinguyennguyen/eagle3-go/wenliang1990/gpt-oss-120b-eagle3-aimo3'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 0
    
    max_judger_calls = 50      
    judger_timeout = 500         

    notebook_limit = 17400
    server_timeout = 300

    session_timeout = 960
    jupyter_timeout = 20
    sandbox_timeout = 3

    stream_interval = 200 # <-- ALIGNED WITH COLAB
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    batch_size = 32

    early_stop_unanimous = 3  
    early_stop = 3
    attempts = 3
    judger_attempts = 3 # <-- 3 PARALLEL JUDGERS REQUESTED
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0 # (Base model diversity, judgers forced to 0.5 below)
    
    save_traces = False              
    trace_dir = '/kaggle/working/traces'
    verbose_traces = False


set_seed(CFG.seed)

In [10]:
set_seed(CFG.seed)

In [11]:
class AIMO3Template:
    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:
        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

In [12]:
class AIMO3Sandbox:
    sys.set_int_max_str_digits(0)
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\nimport numpy\nimport sympy\nimport scipy\nimport scipy.special\n'
            'import itertools\nimport collections\nimport mpmath\nmpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:
        clean_lines =[]
        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts =[]
        stderr_parts =[]
        start_time = time.time()

        while True:
            if time.time() - start_time > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')
                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                stderr_parts.append(self._format_error(content.get('traceback',[])))

            elif msg_type in {'execute_result', 'display_data'}:
                text = content.get('data', {}).get('text/plain')
                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status' and content.get('execution_state') == 'idle':
                break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        self.execute(
            '%reset -f\nimport sys\nsys.set_int_max_str_digits(0)\nsys.setrecursionlimit(20000)\n'
            'import math\nimport numpy\nimport sympy\nimport scipy\nimport scipy.special\n'
            'import itertools\nimport collections\nimport mpmath\nmpmath.mp.dps = 64\n'
        )

    def __del__(self):
        self.close()

In [13]:
class AIMO3Tool:
    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(name='python', description=self._tool_prompt, tools=[])

    def _make_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')
        if channel:
            message = message.with_channel(channel)
        return message

    def process_sync_plus(self, message: Message) -> list[Message]:
        self._ensure_session()
        raw_script = message.content[0].text
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(raw_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return[self._make_response(output, channel=message.channel)]


In [14]:
class AIMO3Solver:
    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.judger_calls = 0
    
    def _preload_model_weights(self) -> None:
        print(f'Loading model weights into OS Page Cache...')
        start_time = time.time()
        
        files_to_load =[]
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
                    
        if hasattr(self.cfg, 'draft_model_path') and os.path.exists(self.cfg.draft_model_path):
            for root, _, files in os.walk(self.cfg.draft_model_path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        files_to_load.append(file_path)
                        total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:
        spec_config_json = f'{{"model":"{self.cfg.draft_model_path}","num_speculative_tokens":3,"method":"eagle3","draft_tensor_parallel_size":1}}'
        cmd =[
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server', 
            '--seed', str(self.cfg.seed), 
            '--model', self.cfg.model_path, 
            '--served-model-name', self.cfg.served_model_name, 
            '--tensor-parallel-size', '1', 
            '--max-num-seqs', str(self.cfg.batch_size), 
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), 
            '--host', '0.0.0.0', 
            '--port', str(self.port), 
            '--dtype', self.cfg.dtype, 
            '--kv-cache-dtype', self.cfg.kv_cache_dtype, 
            '--max-model-len', str(self.cfg.context_tokens), 
            '--stream-interval', str(self.cfg.stream_interval), 
            '--async-scheduling',             
            '--enable-prefix-caching',  
            '--speculative-config', spec_config_json,
            '--swap-space', '16'
        ]
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)
    
    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            if return_code is not None:
                self.log_file.flush()
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
        self.sandbox_pool = queue.Queue()
        def _create_sandbox():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures =[executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        # Better regex from Colab
        for pattern in [r'\\boxed\s*\{(?:.*?[=:])?\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999:
                        return val
                except ValueError:
                    pass
        return None
    
    def _format_conversation_trace(self, conversation, attempt_index) -> str:
        idx_str = attempt_index if isinstance(attempt_index, str) else str(attempt_index + 1)
        if not conversation: return f"--- No conversation recorded for Attempt {idx_str} ---"
        trace_lines =[f"========== TRACE FOR ATTEMPT {idx_str} =========="]

        for msg in conversation.messages:
            role_str = str(getattr(msg.author, 'role', 'UNKNOWN')).replace('Role.', '').upper()
            author_name = getattr(msg.author, 'name', None)
            channel = getattr(msg, 'channel', None)
            recipient = getattr(msg, 'recipient', None)

            text_parts =[]
            if hasattr(msg, 'content') and msg.content:
                for c in (msg.content if isinstance(msg.content, list) else[msg.content]):
                    if hasattr(c, 'text'):
                        text_parts.append(str(c.text))
                    elif type(c).__name__ == 'DeveloperContent':
                        text_parts.append(str(getattr(c, 'instructions', '<Instructions>')))
                    elif type(c).__name__ == 'SystemContent':
                        sys_text = f"[Identity]: {getattr(c, 'model_identity', '')}\n[Effort]: {getattr(c, 'reasoning_effort', '')}"
                        if hasattr(c, 'tools') and c.tools:
                            sys_text += f"\n[Tools]: {getattr(c.tools, 'description', '')}"
                        text_parts.append(sys_text.strip())
                    else:
                        text_parts.append(str(c))

            text = "\n".join(text_parts).strip()
            if role_str == 'USER': trace_lines.append(f"\n[🟢 USER]\n{text}")
            elif role_str == 'SYSTEM': trace_lines.append(f"\n[⚙️ SYSTEM]\n{text}")
            elif role_str == 'DEVELOPER': trace_lines.append(f"\n[🛠️ DEVELOPER]\n{text}")
            elif role_str == 'ASSISTANT':
                if channel == 'analysis': trace_lines.append(f"\n[🧠 ASSISTANT (Channel: analysis) - Thinking]\n{text}")
                elif channel == 'commentary': trace_lines.append(f"\n[⚡ ASSISTANT (Channel: commentary) - Calling Tool: to={recipient}]\n{text}" if recipient else f"\n[💬 ASSISTANT (Channel: commentary) - Preamble]\n{text}")
                elif channel == 'final': trace_lines.append(f"\n[🎯 ASSISTANT (Channel: final) - Final Answer]\n{text}")
                else: trace_lines.append(f"\n[🤖 ASSISTANT]\n{text}")
            elif role_str == 'TOOL' or author_name == 'python':
                trace_lines.append(f"\n[🖥️ TOOL RESPONSE (from={author_name or 'tool'})]\n{text}")
            else: trace_lines.append(f"\n[{role_str}]\n{text}")

        trace_lines.append("\n===========================================")
        return "\n".join(trace_lines)

    def _process_attempt(self, problem: str, system_prompt: str, attempt_index: int, stop_event: threading.Event, deadline: float) -> dict:
        attempt_start_time = time.time()
        conversation = None

        if stop_event.is_set() or time.time() > deadline:
            return {'Attempt': attempt_index + 1, 'Answer': None, 'Python Calls': 0, 'Python Errors': 0, 'Response Length': 0, 'Execution Time': time.time() - attempt_start_time, 'Trace': "", 'Summary': "", 'Seed': self.cfg.seed}
    
        sandbox = None
        python_calls, python_errors, total_tokens = 0, 0, 0
        final_answer = None
        summary_text = ""
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(local_jupyter_timeout=self.cfg.jupyter_timeout, tool_prompt=self.cfg.tool_prompt, sandbox=sandbox)
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(system_prompt, problem, local_tool.tool_config)
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )
    
                try:
                    token_buffer, text_chunks = [],[]
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
                        
                        choice = chunk.choices[0]
                        new_text = choice.text
                        
                        new_tokens = getattr(choice, 'token_ids', None)
                        if new_tokens is None and hasattr(choice, 'model_extra') and choice.model_extra:
                            new_tokens = choice.model_extra.get('token_ids',[])
                        if new_tokens is None:
                            new_tokens =[]
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
                            if answer is not None:
                                # COLAB 42 TRAP FIX
                                if answer == 42:
                                    pass
                                else:
                                    final_answer = answer
                                    break
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    if token_buffer:
                        try:
                            conversation.messages.extend(encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT))
                        except Exception: pass
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
                
                message_text = "".join([str(c.text) for c in last_message.content if hasattr(c, 'text')])
    
                if last_message.channel == 'final':
                    final_answer = self._scan_for_answer(message_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    resp_text = tool_responses[0].content[0].text
    
                    if resp_text.startswith('[ERROR]') or 'Traceback' in resp_text or 'Error:' in resp_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
                else:
                    ans = self._scan_for_answer(message_text)
                    if ans is not None:
                        final_answer = ans
                        break
    
        except Exception as e:
            python_errors += 1
            print(f"\n[CRASH IN ATTEMPT {attempt_index + 1}] {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

            trace_text = self._format_conversation_trace(conversation, attempt_index)
            if getattr(self.cfg, 'save_traces', False):
                os.makedirs(self.cfg.trace_dir, exist_ok=True)
                with open(f"{self.cfg.trace_dir}/q{getattr(self, 'problem_idx', 'X')}_attempt_{attempt_index + 1}.txt", 'w', encoding='utf-8') as f:
                    f.write(trace_text)
    
        return {
            'Attempt': attempt_index + 1, 'Response Length': total_tokens, 'Python Calls': python_calls, 
            'Python Errors': python_errors, 'Answer': final_answer, 'Execution Time': time.time() - attempt_start_time,
            'Trace': trace_text, 'Summary': summary_text, 'Seed': attempt_seed
        }

    def _generate_summary(self, problem: str, trace_text: str, attempt_index: int, attempt_seed: int, deadline: float) -> dict:
        start_time = time.time()
        summary_text = ""
        input_tokens = 0
        output_tokens = 0

        summarizer_user_prompt = (
            f"Here is the original problem:\n{problem}\n\n"
            f"Here is the raw trace of the solver's thought process and code executions:\n"
            f"{trace_text}\n\n"
            f"Based ONLY on the trace above, write a clean, comprehensive and objective summary of the solution. "
            f"Keep it lean but detailed enough for a complete verification by the judger.Judger will compare it with other provided solution summaries"
        )

        sum_system_content = (SystemContent.new()
            .with_model_identity(self.cfg.summarizer_system_prompt)
            .with_reasoning_effort(self.cfg.summarizer_reasoning_effort))

        sum_messages =[
            Message.from_role_and_content(Role.SYSTEM, sum_system_content),
            Message.from_role_and_content(Role.USER, summarizer_user_prompt)
        ]
        fresh_conversation = Conversation.from_messages(sum_messages)

        try:
            sum_prompt_ids = self.encoding.render_conversation_for_completion(fresh_conversation, Role.ASSISTANT)
            input_tokens = len(sum_prompt_ids)
            sum_max_tokens = min(10000, self.cfg.context_tokens - input_tokens)

            if sum_max_tokens > 100:
                sum_stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    max_tokens=sum_max_tokens,
                    prompt=sum_prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )
                
                sum_token_buffer =[]
                try:
                    for chunk in sum_stream:
                        if time.time() > deadline:
                            break
                        if chunk.choices[0].token_ids:
                            sum_token_buffer.extend(chunk.choices[0].token_ids)
                finally:
                    sum_stream.close()

                output_tokens = len(sum_token_buffer)

                if sum_token_buffer:
                    sum_msgs = self.encoding.parse_messages_from_completion_tokens(sum_token_buffer, Role.ASSISTANT)
                    parts =[
                        str(c.text) for m in sum_msgs
                        if getattr(m, 'channel', None) in ('final', None)
                        for c in m.content if hasattr(c, 'text')
                    ]
                    summary_text = "\n".join(parts).strip()
                    fresh_conversation.messages.extend(sum_msgs)
        except Exception as e:
            print(f"Summarization error for attempt {attempt_index}: {e}")

        duration = time.time() - start_time
        is_valid = bool(summary_text.strip())

        sum_trace_log = f"========== SUMMARIZER PROMPT ==========\n{summarizer_user_prompt}\n\n========== SUMMARIZER PROCESS ==========\n"
        sum_trace_log += self._format_conversation_trace(fresh_conversation, attempt_index=f"{attempt_index}_SUMMARIZER")

        if getattr(self.cfg, 'save_traces', False):
            os.makedirs(self.cfg.trace_dir, exist_ok=True)
            with open(f"{self.cfg.trace_dir}/q{getattr(self, 'problem_idx', 'X')}_attempt_{attempt_index}_summarizer_trace.txt", 'w', encoding='utf-8') as f:
                f.write(sum_trace_log)

        return {
            'Summary': summary_text,
            'Summ_In_Tokens': input_tokens,
            'Summ_Out_Tokens': output_tokens,
            'Summ_Time': duration,
            'Summ_Valid': is_valid
        }

# COLAB: Parallel Judger Attempt Method
    def _process_judger_attempt(self, judger_prompt: str, attempt_index: int, attempt_seed: int, stop_event: threading.Event, deadline: float) -> dict:
        start_time = time.time()
        
        # Check if already stopped before starting
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index, 'Answer': None, 'In Tokens': 0, 'Out Tokens': 0,
                'Time (s)': time.time() - start_time, 'Python Calls': 0, 'Python Errors': 0, 'Trace': "--- Cancelled ---"
            }

        input_tokens = 0
        output_tokens = 0
        python_calls = 0
        python_errors = 0
        first_turn = True

        final_answer = None

        encoding = self.encoding
        tool_config = ToolNamespaceConfig(name='python', description=self.cfg.tool_prompt, tools=[])
        messages = self.template.apply_chat_template(self.cfg.system_prompt, judger_prompt, tool_config)
        conversation = Conversation.from_messages(messages)

        sandbox = None
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(local_jupyter_timeout=self.cfg.jupyter_timeout, tool_prompt=self.cfg.tool_prompt, sandbox=sandbox)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)

                if first_turn:
                    input_tokens = len(prompt_ids)
                    first_turn = False

                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens: break

                # FORCING TEMPERATURE = 0.5 FOR JUDGER 
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, max_tokens=max_tokens, prompt=prompt_ids,
                    seed=attempt_seed, stream=True, extra_body={ 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )

                token_buffer, text_chunks = [], []
                try:
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        choice = chunk.choices[0]
                        new_tokens = getattr(choice, 'token_ids', None)
                        if new_tokens is None and hasattr(choice, 'model_extra') and choice.model_extra:
                            new_tokens = choice.model_extra.get('token_ids', [])

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            text_chunks.append(choice.text)

                        if '}' in choice.text:
                            ans = self._scan_for_answer(''.join(text_chunks[-self.cfg.search_tokens:]))
                            if ans is not None:
                                # COLAB 42 TRAP FIX
                                if ans == 42:
                                    pass
                                else:
                                    final_answer = ans
                                    break
                finally:
                    stream.close()

                output_tokens += len(token_buffer)

                if final_answer is not None or stop_event.is_set():
                    if token_buffer:
                        try: conversation.messages.extend(encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT))
                        except Exception: pass
                    break

                if not token_buffer: break
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                message_text = "".join([str(c.text) for c in last_message.content if hasattr(c, 'text')])

                if last_message.channel == 'final':
                    final_answer = self._scan_for_answer(message_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)

                    resp_text = tool_responses[0].content[0].text
                    if resp_text.startswith('[ERROR]') or 'Traceback' in resp_text or 'Error:' in resp_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)
                else:
                    ans = self._scan_for_answer(message_text)
                    if ans is not None:
                        final_answer = ans
                        break

        except Exception as e:
            python_errors += 1
        finally:
            if sandbox:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        execution_time = time.time() - start_time
        trace_text = self._format_conversation_trace(conversation, attempt_index=f"JUDGER_{attempt_index}")

        if getattr(self.cfg, 'save_traces', False):
            os.makedirs(self.cfg.trace_dir, exist_ok=True)
            with open(f"{self.cfg.trace_dir}/q{getattr(self, 'problem_idx', 'X')}_judger_attempt_{attempt_index}.txt", 'w', encoding='utf-8') as f:
                f.write(f"========== JUDGER PROMPT ==========\n{judger_prompt}\n\n========== JUDGER PROCESS ==========\n{trace_text}")

        return {
            'Attempt': attempt_index,
            'Answer': final_answer,
            'In Tokens': input_tokens,
            'Out Tokens': output_tokens,
            'Time (s)': execution_time,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Trace': trace_text
        }
        
    # COLAB: Voting Matrix
    def _decide_final_answer(self, base_results: list, judger_results: list) -> int:
        candidates = set()
        judger_counts = Counter()
        total_counts = Counter()
        token_sums = defaultdict(int)
        token_counts = defaultdict(int)

        # 1. Tally Judger votes
        for res in judger_results:
            ans = res.get('Answer')
            if ans is not None:
                candidates.add(ans)
                judger_counts[ans] += 1
                total_counts[ans] += 1
                token_sums[ans] += res.get('Out Tokens', 0)
                token_counts[ans] += 1

        # 2. Tally Base votes
        for res in base_results:
            ans = res.get('Answer')
            if ans is not None:
                candidates.add(ans)
                total_counts[ans] += 1
                token_sums[ans] += res.get('Response Length', 0)
                token_counts[ans] += 1

        if not candidates:
            return 0

        # 3. Build Scoreboard
        scoring = []
        for ans in candidates:
            avg_tokens = token_sums[ans] / max(1, token_counts[ans])
            scoring.append({
                'Answer': ans,
                'Judger Votes': judger_counts[ans],
                'Total Votes': total_counts[ans],
                'Avg Tokens': avg_tokens
            })

        # 4. HIERARCHY: 1. Judger Votes | 2. Total Votes | 3. Lowest Avg Tokens
        scoring.sort(key=lambda x: (x['Judger Votes'], x['Total Votes'], -x['Avg Tokens']), reverse=True)

        print("\n--- Final Decision Matrix (Smart Voting) ---")
        display(pd.DataFrame(scoring).round({'Avg Tokens': 1}))

        return scoring[0]['Answer']
    
    def solve_problem(self, problem: str) -> int:
        self.problem_idx = getattr(self, 'problem_idx', 0) + 1
        print(f'\nProblem: {problem}\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        budget = max(self.cfg.base_problem_timeout, min(self.cfg.high_problem_timeout, self.cfg.notebook_limit - elapsed_global - max(0, self.problems_remaining - 1) * self.cfg.base_problem_timeout))
        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        detailed_results, valid_answers = [],[]
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        early_stop_unanimous = getattr(self.cfg, 'early_stop_unanimous', 3)
        early_stop_standard = getattr(self.cfg, 'early_stop', 4)

        try:
            futures =[executor.submit(self._process_attempt, user_input, self.cfg.system_prompt, i, stop_event, deadline) for i in range(self.cfg.attempts)]
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    if getattr(self.cfg, 'verbose_traces', False):
                        print(f"\n>>> TRACE FINISHED FOR ATTEMPT {result['Attempt']}\n{result['Trace']}")

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                        
                        consensus_reached = False
                        
                        if len(valid_answers) > 0:
                            if len(valid_answers) >= early_stop_unanimous and len(set(valid_answers)) == 1:
                                consensus_reached = True
                            else:
                                answer_counts = Counter(valid_answers)
                                if any(count >= early_stop_standard for count in answer_counts.values()):
                                    consensus_reached = True
                                
                        if consensus_reached:
                            stop_event.set()
                            for f in futures: f.cancel()
                            break
                            
                except Exception as exc: 
                    print(f'Future failed: {exc}')
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_df = pd.DataFrame(detailed_results)
            results_df['Execution Time'] = results_df['Execution Time'].round(2)
            results_df['Answer'] = results_df['Answer'].astype('Int64')

            drop_cols = [c for c in ['Trace', 'Summary', 'Seed'] if c in results_df.columns]
            display(results_df.drop(columns=drop_cols) if drop_cols else results_df)

        if not valid_answers:
            print('\nResult: 0\n')
            return 0

        # --- Base consensus check ---
        judger_detailed_results = []
        final_consensus_reached = False
        max_votes_achieved = 0
        
        if valid_answers:
            answer_counts = Counter(valid_answers)
            max_votes_achieved = max(answer_counts.values())
            
            if len(valid_answers) >= early_stop_unanimous and len(set(valid_answers)) == 1:
                final_consensus_reached = True
            elif max_votes_achieved >= early_stop_standard:
                final_consensus_reached = True

        if not final_consensus_reached and len(set(valid_answers)) > 1:
            print(f"\n{'='*50}")
            print(f" NO CONSENSUS REACHED (Max Votes: {max_votes_achieved}). ")
            
            elapsed_global_current = time.time() - self.notebook_start_time
            global_time_left = self.cfg.notebook_limit - elapsed_global_current
            required_reserve = max(0, self.problems_remaining - 1) * self.cfg.base_problem_timeout
            available_for_judger = global_time_left - required_reserve
            
            max_judger_calls = getattr(self.cfg, 'max_judger_calls', 0)
            judger_timeout = getattr(self.cfg, 'judger_timeout', 300)
            
            if self.judger_calls >= max_judger_calls:
                print(f"=== BYPASSING JUDGER: Max allowed calls ({max_judger_calls}) reached. ===")
            elif available_for_judger < 60:
                print(f"=== BYPASSING JUDGER: Insufficient global time buffer ({available_for_judger:.1f}s left). ===")
            else:
                self.judger_calls += 1
                print(f" TRIGGERING {self.cfg.judger_attempts} PARALLEL JUDGER NODES (Call {self.judger_calls} of {max_judger_calls}) ")
                print(f"{'='*50}")

                judger_budget = min(judger_timeout, available_for_judger)
                judger_deadline = time.time() + judger_budget
                
                print("\n>>> Generating Summaries lazily for Judger evaluation...")
                summarizer_futures = {}
                with ThreadPoolExecutor(max_workers=self.cfg.workers) as sum_executor:
                    for res in detailed_results:
                        if res['Answer'] is not None:
                            future = sum_executor.submit(
                                self._generate_summary, problem, res['Trace'], res['Attempt'], res.get('Seed', self.cfg.seed), judger_deadline
                            )
                            summarizer_futures[future] = res

                    sum_metrics =[]
                    for future in as_completed(summarizer_futures):
                        res = summarizer_futures[future]
                        try:
                            summ_info = future.result()
                            res['Summary'] = summ_info['Summary']
                            sum_metrics.append({
                                'Attempt': res['Attempt'],
                                'Answer': res['Answer'],
                                'In Tokens': summ_info['Summ_In_Tokens'],
                                'Out Tokens': summ_info['Summ_Out_Tokens'],
                                'Time (s)': round(summ_info['Summ_Time'], 2),
                                'Valid': summ_info['Summ_Valid']
                            })
                        except Exception as exc:
                            print(f"Summarizer future failed for attempt {res['Attempt']}: {exc}")

                if sum_metrics:
                    print("\n--- Summarizer Node Metrics ---")
                    display(pd.DataFrame(sum_metrics))

                # BUILD KAGGLE-SPECIFIC JUDGER PROMPT
                summaries_text = ""
                valid_results = [r for r in detailed_results if r['Answer'] is not None and r.get('Summary')]
                for res in valid_results:
                    summaries_text += f"### Candidate from Attempt {res['Attempt']} (Proposed Answer: {res['Answer']}) ###\n{res['Summary']}\n\n"

                if summaries_text.strip():
                    judger_prompt = (
                        f"You are an expert mathematical judge. Below is a math problem, followed by several candidate "
                        f"solution summaries from different ai solvers.\n\n"
                        f"Problem:\n{problem_text}\n\n"
                        f"{self.cfg.preference_prompt}\n\n"
                        f"Summaries:\n{summaries_text}\n"
                        f"Your task:\n"
                        f"1. Carefully read the question understand it's intend and evaluate the mathematical reasoning in each candidate summary.\n"
                        f"3. Write Python code when nedeed to explicitly test divergent claims presented in summaries to prove which one is correct.\n"
                        f"4. Provide your final correct answer inside \\boxed{{}} (e.g. \\boxed{{42}}).\n\n"
                        f"Ensure your final answer is a non-negative integer."
                    )
                    
                    # RUN PARALLEL JUDGERS
                    judger_stop_event = threading.Event()
                    valid_judger_answers = []
                    
                    with ThreadPoolExecutor(max_workers=self.cfg.workers) as j_exec:
                        j_futures = [
                            j_exec.submit(
                                self._process_judger_attempt, 
                                judger_prompt, 
                                i+1, 
                                self.cfg.seed+1000+i, 
                                judger_stop_event, 
                                judger_deadline
                            ) for i in range(self.cfg.judger_attempts)
                        ]
                        
                        # Dynamically calculates simple majority (e.g. 3 attempts -> needs 2, 5 attempts -> needs 3)
                        majority_threshold = (self.cfg.judger_attempts // 2) + 1
                        
                        for f in as_completed(j_futures):
                            try: 
                                res = f.result()
                                judger_detailed_results.append(res)
                                ans = res.get('Answer')
                                
                                if ans is not None:
                                    valid_judger_answers.append(ans)
                                    ans_counts = Counter(valid_judger_answers)
                                    
                                    # If majority consensus is reached, kill remaining jobs and break
                                    if any(count >= majority_threshold for count in ans_counts.values()):
                                        judger_stop_event.set()
                                        for pf in j_futures: 
                                            pf.cancel()
                                        break
                                        
                            except Exception: 
                                pass

                    if judger_detailed_results:
                        print("\n--- Judger Node Metrics ---")
                        j_df = pd.DataFrame(judger_detailed_results)
                        display(j_df.drop(columns=['Trace'], errors='ignore'))
                        
        else:
            print("\n=== CONSENSUS REACHED: Bypassing Judger ===")

        # FINAL DECISION VIA SMART VOTING
        print("\n" + "="*50)
        print("=== FINAL DECISION OVERVIEW ===")
        final_answer = self._decide_final_answer(detailed_results, judger_detailed_results)
        print(f"SUBMITTED ANSWER              : {final_answer}")
        print("="*50 + "\n")

        return final_answer
            
    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass

In [15]:
solver = AIMO3Solver(CFG)

Loading model weights into OS Page Cache...
Processed 42 files (65.89 GB) in 84.44 seconds.

Waiting for vLLM server...
Server is ready (took 168.26 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.47 seconds.



In [16]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})



In [17]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )


Problem: What is $0\times10$?

Budget: 900.00 seconds | Deadline: 1775084758.41



,Attempt,Response Length,Python Calls,Python Errors,Answer,Execution Time
0,3,139,0,0,0,0.97
1,2,201,0,0,0,1.27
2,1,200,0,0,0,1.42



=== CONSENSUS REACHED: Bypassing Judger ===

=== FINAL DECISION OVERVIEW ===

--- Final Decision Matrix (Smart Voting) ---


,Answer,Judger Votes,Total Votes,Avg Tokens
0,0,0,3,180.0


SUBMITTED ANSWER              : 0


Problem: Solve $4+x=4$ for $x$.

Budget: 900.00 seconds | Deadline: 1775084760.06



...

SUBMITTED ANSWER              : 0


Problem: What is $1-1$?

Budget: 900.00 seconds | Deadline: 1775084761.49



,Attempt,Response Length,Python Calls,Python Errors,Answer,Execution Time
0,2,141,0,0,0,0.91
1,1,166,0,0,0,1.05
2,3,201,0,0,0,1.26



=== CONSENSUS REACHED: Bypassing Judger ===

=== FINAL DECISION OVERVIEW ===

--- Final Decision Matrix (Smart Voting) ---


,Answer,Judger Votes,Total Votes,Avg Tokens
0,0,0,3,169.3


SUBMITTED ANSWER              : 0



In [18]:
import csv

data = [
    ["id", "problem", "answer"],

    # --- TMO 2025 Problems ---
    ["tmo_2025_01", "In a triangle $ABC$, let $H$ be the orthocenter and $G$ be the centroid. If $|BH| = 3\\sqrt{2}$, $|CH| = 6$, and \\angle BHC = 135^\\circ$, what is $|GH|$?", "2"],
    ["tmo_2025_12", "Let $k$ be a positive integer. One of the numbers $1, 2, \\dots, k$ is written in each unit square of a $4 \\times 4$ chessboard. If for every pair $(m,n)$ satisfying $1 \\le m < n \\le k$, there exists a row or a column containing both $m$ and $n$, what is the maximum possible value of $k$?", "8"],

    ["tmo_2025_02", "How many positive integers $n < 2025$ are there such that the number $1^3 + 2^3 + \\dots + n^3$ is divisible by $2025$?", "179"],
    ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],
    ["imo_2025_01_mod", "A line in the plane is called sunny if it is not parallel to any of the $x$-axis, the $y$-axis, and the line $x + y = 0$. Let $n \\ge 3$ be a given integer. Determine the sum of all possible nonnegative integers $k$ such that there exist $n$ distinct lines in the plane satisfying both of the following: for all positive integers $a$ and $b$ with $a + b \\le n + 1$, the point $(a, b)$ is on at least one of the lines; and exactly $k$ of the $n$ lines are sunny.", "4"],

    ["tmo_2025_03", "Let $m$ and $n$ be integers. How many polynomials $f(x) = x^2 + mx + n$ satisfy the condition $f(f(360)) = 0$?", "48"],
    ["tmo_2025_04", "A person with 6 different hats wears one of these hats each day for 6 consecutive days. This person wears different hats on any two consecutive days, but wears the same hat on the first and the last day. In how many different sequences can this person wear the hats over these 6 days?", "3120"],
        ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],

    ["tmo_2025_05", "In a cyclic pentagon $ABCDE$, the conditions $AB \\parallel CE$ and $AC \\parallel DE$ are satisfied. Let $F$ be the intersection point of the lines $AB$ and $CD$. If $|CF| = 8$, $|CD| = 12$, and $|DE| = 30$, what is $|AC|$?", "20"],
    ["tmo_2025_06", "For how many of the numbers $n \\in \\{195, 196, 197, 198, 199, 200\\}$ is the number $1^n + 2^{n-1} + 3^{n-2} + \\dots + n^1$ divisible by 3?", "4"],
    ["imo_2025_05_mod", "Alice and Bazza are playing a two-player game whose rules depend on a positive real number $\\lambda$ which is known to both players. On the $n^{\\text{th}}$ turn of the game (starting with $n = 1$) the following happens: If $n$ is odd, Alice chooses a nonnegative real number $x_n$ such that $x_1 + x_2 + \\dots + x_n \\le \\lambda n$. If $n$ is even, Bazza chooses a nonnegative real number $x_n$ such that $x_1^2 + x_2^2 + \\dots + x_n^2 \\le n$. If a player cannot choose a suitable number $x_n$, the game ends and the other player wins. If the game goes on forever, neither player wins. All chosen numbers are known to both players. There is a critical threshold $\\lambda_c$ such that Alice has a winning strategy if $\\lambda > \\lambda_c$, and Bazza has a winning strategy if $0 < \\lambda < \\lambda_c$. If $\\lambda_c$ can be written in the form $\\frac{1}{\\sqrt{m}}$ for some integer $m$, what is the value of $m$?", "2"],

    ["tmo_2025_07", "For how many positive integers $a$ does the polynomial $P(x) = x^6 - 6x^5 + 12x^4 - ax^3 + 12x^2 - 6x + 1$ have no positive real roots?", "11"],
    ["tmo_2025_10", "How many ordered pairs of positive integers $(a,b)$ satisfy the conditions $\\gcd(a,b) = 22!$ and $\\text{lcm}(a,b) = 33!$?", "512"],
    ["tmo_2025_12", "Let $k$ be a positive integer. One of the numbers $1, 2, \\dots, k$ is written in each unit square of a $4 \\times 4$ chessboard. If for every pair $(m,n)$ satisfying $1 \\le m < n \\le k$, there exists a row or a column containing both $m$ and $n$, what is the maximum possible value of $k$?", "8"],
    ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],

    ["tmo_2025_13", "In a triangle $ABC$, let $|AB| = 2$ and $|AC| = 1$. A point $M$ located on the opposite side of the line $AB$ from $C$ satisfies $\\angle MAB = 90^\\circ$ and $|MA| = |AB|$. A point $N$ located on the opposite side of the line $AC$ from $B$ satisfies $\\angle NAC = 90^\\circ$ and $|NA| = |AC|$. Let $O$ be the circumcenter of the triangle $MNA$. If the lines $OA$ and $BC$ intersect at a point $D$, what is the ratio $\\frac{|BD|}{|CD|}$?", "4"],
    ["tmo_2025_14", "For how many positive integers $n \\le 2025$ do there not exist positive integers $x$ and $y$ such that $3^x - 5^y$ is divisible by $n$?", "945"],
    ["tmo_2025_12", "Let $k$ be a positive integer. One of the numbers $1, 2, \\dots, k$ is written in each unit square of a $4 \\times 4$ chessboard. If for every pair $(m,n)$ satisfying $1 \\le m < n \\le k$, there exists a row or a column containing both $m$ and $n$, what is the maximum possible value of $k$?", "8"],

    ["tmo_2025_15", "A sequence of real numbers $a_0, a_1, a_2, \\dots$ is defined by $a_0 = 3$ and for all $n \\ge 1$,\n\\begin{equation*}\n\\frac{a_n + 1}{n} = \\frac{a_{n-1}^2}{n} + 2a_{n-1} + n - 1\n\\end{equation*}\nLet $k$ be a positive integer. What is the minimum possible value of the expression $|a_{2025} - 2^k|$?", "2026"],
    ["tmo_2025_19", "Let $k$ be an integer. How many different solutions are there to the equation\n\\begin{equation*}\n\\left\\lfloor \\frac{k+1}{2025} \\right\\rfloor + \\left\\lfloor \\frac{k+2}{2025} \\right\\rfloor + \\dots + \\left\\lfloor \\frac{k+2024}{2025} \\right\\rfloor = 2025!\n\\end{equation*}\n(For a real number $x$, $\\lfloor x \\rfloor$ denotes the greatest integer not exceeding $x$.)", "2"],
        ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],

    ["tmo_2025_20", "For how many of the pairs $(m,n) \\in \\{(32, 33), (20, 25), (10, 40), (19, 21), (77, 99)\\}$ can an $m \\times n$ chessboard be colored such that each unit square is painted either red or blue, and for every unit square, the number of its adjacent unit squares (sharing a common edge or a common vertex with it) that have the same color as itself is an odd number?", "3"],
    ["imo_2025_05_mod", "Alice and Bazza are playing a two-player game whose rules depend on a positive real number $\\lambda$ which is known to both players. On the $n^{\\text{th}}$ turn of the game (starting with $n = 1$) the following happens: If $n$ is odd, Alice chooses a nonnegative real number $x_n$ such that $x_1 + x_2 + \\dots + x_n \\le \\lambda n$. If $n$ is even, Bazza chooses a nonnegative real number $x_n$ such that $x_1^2 + x_2^2 + \\dots + x_n^2 \\le n$. If a player cannot choose a suitable number $x_n$, the game ends and the other player wins. If the game goes on forever, neither player wins. All chosen numbers are known to both players. There is a critical threshold $\\lambda_c$ such that Alice has a winning strategy if $\\lambda > \\lambda_c$, and Bazza has a winning strategy if $0 < \\lambda < \\lambda_c$. If $\\lambda_c$ can be written in the form $\\frac{1}{\\sqrt{m}}$ for some integer $m$, what is the value of $m$?", "2"],

    ["tmo_2025_21", "The inscribed circle of a right-angled triangle $ABC$ with $\\angle B = 90^\\circ$ is tangent to the sides $AB$ and $AC$ at points $D$ and $E$, respectively. Let $F$ be the intersection point of the lines $ED$ and $BC$. If $|FD| = 25$ and $|DE| = 24$, what is $|AE|$?", "20"],
    ["imo_2025_01_mod", "A line in the plane is called sunny if it is not parallel to any of the $x$-axis, the $y$-axis, and the line $x + y = 0$. Let $n \\ge 3$ be a given integer. Determine the sum of all possible nonnegative integers $k$ such that there exist $n$ distinct lines in the plane satisfying both of the following: for all positive integers $a$ and $b$ with $a + b \\le n + 1$, the point $(a, b)$ is on at least one of the lines; and exactly $k$ of the $n$ lines are sunny.", "4"],

    ["tmo_2025_22", "Let $f(n)$ be the greatest odd divisor of a positive integer $n$. What is the sum $f(25) + f(26) + f(27) + \\dots + f(200)$?", "13150"],
    ["tmo_2025_23", "If positive real numbers $x$, $y$, and $z$ satisfy the system of equations\n\\begin{equation*}\n\\begin{aligned}\n\\frac{y^2}{z} + \\frac{zx+x^2}{2y+z} &= 2x \\\\\\\\\n\\frac{x^2}{z} + \\frac{zy+2y^2}{x+z} &= 9y \\\\\\\\\n\\frac{y^2}{x} + \\frac{x^2}{y} &= 9z\n\\end{aligned}\n\\end{equation*}\nwhat is the value of $\\frac{y}{x} + \\frac{z}{y} + \\frac{x}{z}$?", "5"],
    ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],

    ["tmo_2025_26", "Let $p$ be a prime number, and let $f(p)$ be the number of integers $x$ that satisfy the condition $x + p^2 \\mid x^3 + p^4$. For how many of the numbers $m \\in \\{150, 160, 170, 180, 190\\}$ does there exist a prime number $p$ such that $f(p) = m$?", "2"],
    ["tmo_2025_27", "Initially, there is 1 liter of a homogeneous mixture in a blue jar consisting of $99\\%$ pomegranate juice and $1\\%$ orange juice, and 2 liters of a homogeneous mixture in a green jar consisting of $1\\%$ pomegranate juice and $99\\%$ orange juice. In each step, first, 1 liter of mixture is transferred from the green jar to the blue jar and mixed until it is homogeneous. Then, 1 liter of mixture is transferred from the blue jar back to the green jar and mixed until it is homogeneous. After at least how many steps will the absolute difference between the percentage of pomegranate juice in the blue jar and the percentage of pomegranate juice in the green jar be strictly less than $0.1\\%$?", "5"],
    ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],

    ["tmo_2025_28", "25 points are marked on a circle with a circumference of 25 units such that these points are the vertices of a regular 25-gon, and $k$ of these points are colored red. If, regardless of how this coloring is done, there are always two red points such that the length of the shorter arc between them is 5 units, what is the minimum possible value of $k$?", "11"],
    ["tmo_2025_30", "How many integers $n$ are there such that the expression $n^4 - 5n^3 + 26n^2 - 41n + 19$ is equal to an integer power of a prime number?", "3"],
    ["tmo_2025_32", "A $1 \\times N$ chessboard, with initially no unit squares colored, is drawn on a board. Two players, $A$ and $B$, play a game by taking turns making moves, with player $A$ starting the game. In their turn, player $A$ colors an uncolored unit square red, and player $B$ colors an uncolored unit square blue in their turn. Neither player is allowed to make a move such that two unit squares sharing a common edge have the same color. The player who cannot make a move loses the game. If the game is played exactly once for each of the values $N \\in \\{2023, 2024, 2025, 2026, 2027\\}$, how many of these games can player $A$ guarantee to win?", "0"],

    # --- New Reference Problems ---
    ["reference_01", "Alice and Bob are each holding some integer number of sweets. Alice says to Bob: \"If we each added the number of sweets we're holding to our (positive integer) age, my answer would be double yours. If we took the product, then my answer would be four times yours.\" Bob replies: \"Why don't you give me five of your sweets because then both our sum and product would be equal.\" What is the product of Alice and Bob's ages?", "50"],
    ["reference_02", "A $500 \\times 500$ square is divided into $k$ rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\\mathcal{K}$. What is the remainder when $\\mathcal{K}$ is divided by $10^5$?", "520"],
    ["reference_03", "Let $ABC$ be an acute-angled triangle with integer side lengths and $AB < AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD = AE = AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \\neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a = BC$, $b = CA$, and $c = AB$. Find the remainder when $abc$ is divided by $10^5$.", "336"],
    ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],

    ["reference_04", "Let $f \\colon \\mathbb{Z}_{\\ge 1} \\to \\mathbb{Z}_{\\ge 1}$ be a function such that for all positive integers $m$ and $n$,\n\\begin{equation*}\nf(m) + f(n) = f(m + n + mn).\n\\end{equation*}\nAcross all functions $f$ such that $f(n) \\le 1000$ for all $n \\le 1000$, how many different values can $f(2024)$ take?", "580"],
    ["reference_05", "A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of 20 rounds with each runner starting with a score of 0. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points.\n\nAt the end of the tournament, we rank the competitors according to their scores. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^5$?", "21818"],
    ["imo_2025_01_mod", "A line in the plane is called sunny if it is not parallel to any of the $x$-axis, the $y$-axis, and the line $x + y = 0$. Let $n \\ge 3$ be a given integer. Determine the sum of all possible nonnegative integers $k$ such that there exist $n$ distinct lines in the plane satisfying both of the following: for all positive integers $a$ and $b$ with $a + b \\le n + 1$, the point $(a, b)$ is on at least one of the lines; and exactly $k$ of the $n$ lines are sunny.", "4"],

    ["reference_06", "Define a function $f \\colon \\mathbb{Z}_{\\ge 1} \\to \\mathbb{Z}_{\\ge 1}$ by\n\\begin{equation*}\nf(n) = \\sum_{i=1}^n \\sum_{j=1}^n j^{1024} \\left\\lfloor \\frac{1}{j} + \\frac{n-i}{n} \\right\\rfloor.\n\\end{equation*}\nLet $M = 2 \\cdot 3 \\cdot 5 \\cdot 7 \\cdot 11 \\cdot 13$ and let $N = f(M^{15}) - f(M^{15} - 1)$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?", "32951"],
    ["reference_07", "Let $ABC$ be a triangle with $AB \\neq AC$, circumcircle $\\Omega$, and incircle $\\omega$. Let the contact points of $\\omega$ with $BC$, $CA$, and $AB$ be $D$, $E$, and $F$, respectively. Let the circumcircle of $AFE$ meet $\\Omega$ at $K$ and let the reflection of $K$ in $EF$ be $K'$. Let $N$ denote the foot of the perpendicular from $D$ to $EF$. The circle tangent to line $BN$ and passing through $B$ and $K$ intersects $BC$ again at $T \\neq B$.\n\nLet sequence $(F_n)_{n \\ge 0}$ be defined by $F_0 = 0$, $F_1 = 1$ and for $n \\ge 2$, $F_n = F_{n-1} + F_{n-2}$. Call $ABC$ $n$\\emph{-tastic} if $BD = F_n$, $CD = F_{n+1}$, and $KNK'B$ is cyclic. Across all $n$-tastic triangles, let $a_n$ denote the maximum possible value of $\\frac{CT \\cdot NB}{BT \\cdot NE}$. Let $\\alpha$ denote the smallest real number such that for all sufficiently large $n$, $a_{2n} < \\alpha$. Given that $\\alpha = p + \\sqrt{q}$ for rationals $p$ and $q$, what is the remainder when $\\left\\lfloor p^{q^p} \\right\\rfloor$ is divided by $99991$?", "57447"],
    ["reference_08", "On a blackboard, Ken starts off by writing a positive integer $n$ and then applies the following move until he first reaches $1$. Given that the number on the board is $m$, he chooses a base $b$, where $2 \\le b \\le m$, and considers the unique base-$b$ representation of $m$,\n\\begin{equation*}\nm = \\sum_{k=0}^\\infty a_k \\cdot b^k\n\\end{equation*}\nwhere $a_k$ are non-negative integers and $0 \\le a_k < b$ for each $k$. Ken then erases $m$ on the blackboard and replaces it with $\\sum_{k=0}^\\infty a_k$.\n\nAcross all choices of $1 \\le n \\le 10^{10^5}$, the largest possible number of moves Ken could make is $M$. What is the remainder when $M$ is divided by $10^5$?", "32193"],
    ["tmo_2025_12", "Let $k$ be a positive integer. One of the numbers $1, 2, \\dots, k$ is written in each unit square of a $4 \\times 4$ chessboard. If for every pair $(m,n)$ satisfying $1 \\le m < n \\le k$, there exists a row or a column containing both $m$ and $n$, what is the maximum possible value of $k$?", "8"],

    ["reference_09", "Let $\\mathcal{F}$ be the set of functions $\\alpha \\colon \\mathbb{Z} \\to \\mathbb{Z}$ for which there are only finitely many $n \\in \\mathbb{Z}$ such that $\\alpha(n) \\neq 0$.\n\nFor two functions $\\alpha$ and $\\beta$ in $\\mathcal{F}$, define their product $\\alpha \\star \\beta$ to be $\\sum_{n \\in \\mathbb{Z}} \\alpha(n) \\cdot \\beta(n)$. Also, for $n \\in \\mathbb{Z}$, define a shift operator $S_n \\colon \\mathcal{F} \\to \\mathcal{F}$ by $S_n(\\alpha)(t) = \\alpha(t+n)$ for all $t \\in \\mathbb{Z}$.\n\nA function $\\alpha \\in \\mathcal{F}$ is called \\emph{shifty} if\n\\begin{itemize}\n\\item $\\alpha(m) = 0$ for all integers $m < 0$ and $m > 8$ and\n\\item There exists $\\beta \\in \\mathcal{F}$ and integers $k \\neq l$ such that for all $n \\in \\mathbb{Z}$\n\\begin{equation*}\nS_n(\\alpha) \\star \\beta = \\begin{cases} 1 & n \\in \\{k,l\\} \\\\\\\\ 0 & n \\notin \\{k,l\\} \\end{cases}.\n\\end{equation*}\n\\end{itemize}\nHow many shifty functions are there in $\\mathcal{F}$?", "160"],
    ["reference_10", "Let $n \\ge 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M = 3^{2025!}$ and for a non-negative integer $c$ define\n\\begin{equation*}\ng(c) = \\frac{1}{2025!} \\left\\lfloor \\frac{2025! f(M+c)}{M} \\right\\rfloor.\n\\end{equation*}\nWe can write\n\\begin{equation*}\ng(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = \\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?", "8687"],
        # Reformulated Problem 1
    ["imo_2025_01_mod", "A line in the plane is called sunny if it is not parallel to any of the $x$-axis, the $y$-axis, and the line $x + y = 0$. Let $n \\ge 3$ be a given integer. Determine the sum of all possible nonnegative integers $k$ such that there exist $n$ distinct lines in the plane satisfying both of the following: for all positive integers $a$ and $b$ with $a + b \\le n + 1$, the point $(a, b)$ is on at least one of the lines; and exactly $k$ of the $n$ lines are sunny.", "4"],

    # Problem 3
    ["imo_2025_03", "Let $\\mathbb{N}$ denote the set of positive integers. A function $f \\colon \\mathbb{N} \\to \\mathbb{N}$ is said to be bonza if $f(a)$ divides $b^a - f(b)^{f(a)}$ for all positive integers $a$ and $b$. Determine the smallest real constant $c$ such that $f(n) \\le cn$ for all bonza functions $f$ and all positive integers $n$.", "4"],

    # Reformulated Problem 5
    ["imo_2025_05_mod", "Alice and Bazza are playing a two-player game whose rules depend on a positive real number $\\lambda$ which is known to both players. On the $n^{\\text{th}}$ turn of the game (starting with $n = 1$) the following happens: If $n$ is odd, Alice chooses a nonnegative real number $x_n$ such that $x_1 + x_2 + \\dots + x_n \\le \\lambda n$. If $n$ is even, Bazza chooses a nonnegative real number $x_n$ such that $x_1^2 + x_2^2 + \\dots + x_n^2 \\le n$. If a player cannot choose a suitable number $x_n$, the game ends and the other player wins. If the game goes on forever, neither player wins. All chosen numbers are known to both players. There is a critical threshold $\\lambda_c$ such that Alice has a winning strategy if $\\lambda > \\lambda_c$, and Bazza has a winning strategy if $0 < \\lambda < \\lambda_c$. If $\\lambda_c$ can be written in the form $\\frac{1}{\\sqrt{m}}$ for some integer $m$, what is the value of $m$?", "2"]
]


with open("tmo_2025_aimo_dataset.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file, quoting=csv.QUOTE_MINIMAL)
    writer.writerows(data)

print("File 'tmo_2025_aimo_dataset.csv' created successfully with total problems!", len(data))


File 'tmo_2025_aimo_dataset.csv' created successfully with total problems! 51


In [19]:
import pandas as pd
import gc

# 1. Load your newly created dataset
df = pd.read_csv("tmo_2025_aimo_dataset.csv")

# 2. IMPORTANT: Update the solver's problem count! 
# The original code expects 50 problems and divides its 9-hour time limit accordingly. 
# Changing this to 10 ensures the solver gives maximum allowed time to your hard problems.
solver.problems_remaining = len(df)

results = []

print("\n" + "="*50)
print(f"STARTING EXPERIMENT ON {len(df)} CUSTOM PROBLEMS")
print("="*50 + "\n")

# 3. Loop through your CSV
for index, row in df.iterrows():
    problem_id = row['id']
    problem_text = row['problem']
    expected_answer = row['answer']
    
    print(f"\n--- Solving Problem {index + 1}/{len(df)}: {problem_id} ---")
    
    # Memory management identical to the original script
    gc.collect()
    gc.disable()
    
    # Generate the prediction
    predicted_answer = solver.solve_problem(problem_text)
    
    gc.enable()
    gc.collect()
    
    # Check correctness (cast both to strings to avoid type mismatch bugs)
    is_correct = str(expected_answer) == str(predicted_answer)
    
    # Store and display intermediate result
    results.append({
        'id': problem_id,
        'expected': expected_answer,
        'predicted': predicted_answer,
        'correct': is_correct
    })
    
    print(f">>> Expected: {expected_answer} | Predicted: {predicted_answer} | Correct: {is_correct}\n")

# 4. Final summary
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("=== FINAL EXPERIMENT SUMMARY ===")
print("="*50)
display(results_df)

accuracy = results_df['correct'].mean() * 100
print(f"\nOverall Accuracy: {accuracy:.2f}% ({results_df['correct'].sum()}/{len(df)} correct)")



STARTING EXPERIMENT ON 50 CUSTOM PROBLEMS


--- Solving Problem 1/50: tmo_2025_01 ---

Problem: In a triangle $ABC$, let $H$ be the orthocenter and $G$ be the centroid. If $|BH| = 3\sqrt{2}$, $|CH| = 6$, and \angle BHC = 135^\circ$, what is $|GH|$?

Budget: 900.00 seconds | Deadline: 1775084763.05



,Attempt,Response Length,Python Calls,Python Errors,Answer,Execution Time
0,2,3456,1,0,2,18.30
1,1,3814,2,0,2,19.70
2,3,4243,4,0,2,21.46



=== CONSENSUS REACHED: Bypassing Judger ===

=== FINAL DECISION OVERVIEW ===

--- Final Decision Matrix (Smart Voting) ---


,Answer,Judger Votes,Total Votes,Avg Tokens
0,2,0,3,3837.7


SUBMITTED ANSWER              : 2

>>> Expected: 2 | Predicted: 2 | Correct: True


--- Solving Problem 2/50: tmo_2025_12 ---

Problem: Let $k$ be a positive integer. One of the numbers $1, 2, \dots, k$ is written in each unit square of a $4 \times 4$ chessboard. If for every pair $(m,n)$ satisfying $1 \le m < n \le k$, there exists a row or a column containing both $m$ and $n$, what is the maximum possible value of $k$?

Budget: 900.00 seconds | Deadline: 1775084784.74



...


=== CONSENSUS REACHED: Bypassing Judger ===

=== FINAL DECISION OVERVIEW ===

--- Final Decision Matrix (Smart Voting) ---


,Answer,Judger Votes,Total Votes,Avg Tokens
0,2,0,3,42035.3


SUBMITTED ANSWER              : 2

>>> Expected: 2 | Predicted: 2 | Correct: True


=== FINAL EXPERIMENT SUMMARY ===


,id,expected,predicted,correct
0,tmo_2025_01,2,2,True
1,tmo_2025_12,8,8,True
2,tmo_2025_02,179,179,True
3,reference_10,8687,8687,True
4,imo_2025_01_mod,4,4,True
5,tmo_2025_03,48,48,True
6,tmo_2025_04,3120,3120,True
7,reference_10,8687,14,False
8,tmo_2025_05,20,20,True
9,tmo_2025_06,4,4,True



Overall Accuracy: 92.00% (46/50 correct)
